# DanceMatch3 — kostenloser Trainingslauf in Google Colab

Dieses Notebook umgeht GitHub Actions. Es verwendet die bestehende Baseline aus dem Branch `feature/dance-style-training` und läuft im kostenlosen Colab-Runtime, soweit dein Google-Konto dafür Zugriff hat. **Kein GitHub-Actions-Lauf und kein GitHub-Billing nötig.**\n\nFür Discofox, Bachata und Salsa kann optional ein ZIP mit eigenen oder anderweitig rechtmässig nutzbaren Audioausschnitten ergänzt werden. Die Musikdateien werden nicht ins GitHub-Repository hochgeladen. Ohne Zusatzdaten ist das Modell ausdrücklich kein vollständiges Neun-Klassen-Modell.

Hinweis: Das ist ein erster Forschungs-Baseline-Test, noch kein produktionsreifes Modell für alle neun Tanzstile. Bachata und Discofox brauchen zusätzliche, sauber gelabelte Trainingsdaten.

In [ ]:
%cd /content
import os, pathlib, shutil
os.chdir('/content')
os.environ['PWD'] = '/content'
REPO_PATH = pathlib.Path('/content/DanceMatch3')
CACHE_DIR = pathlib.Path('/content/dancematch_hf_cache')

# Alte DanceMatch-/MuCUE-Caches aus einem abgebrochenen, nicht gestreamten Lauf gezielt entfernen.
if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
legacy_dataset_cache = pathlib.Path.home() / '.cache' / 'huggingface' / 'datasets'
if legacy_dataset_cache.exists():
    for p in legacy_dataset_cache.glob('*MuCUE*'):
        shutil.rmtree(p, ignore_errors=True)
legacy_hub_cache = pathlib.Path.home() / '.cache' / 'huggingface' / 'hub' / 'datasets--Yi3852--MuCUE'
if legacy_hub_cache.exists():
    shutil.rmtree(legacy_hub_cache, ignore_errors=True)

# Alte Installations-Caches vor dem Start entfernen.
!apt-get clean
!pip cache purge >/dev/null 2>&1 || true
total, used, free = shutil.disk_usage('/content')
print(f'Freier Colab-Speicher vor Installation: {free / (1024**3):.2f} GiB von {total / (1024**3):.2f} GiB')
if free < 1.5 * (1024**3):
    raise RuntimeError('Weniger als 1.5 GiB frei. Die Runtime wurde vorsorglich gestoppt, damit sie nicht voll läuft. Bitte Colab-Runtime neu starten und erneut ausführen.')

# Systembibliotheken und kompatible Python-Abhängigkeiten installieren.
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1
!pip -q install 'datasets>=3.0,<4' 'librosa>=0.10.2,<1' 'numpy>=1.26,<2.3' 'scikit-learn>=1.4,<2' 'joblib>=1.3,<2' 'fsspec==2025.3.0' 'gcsfs==2025.3.0'
import datasets, fsspec
print('datasets:', datasets.__version__, '| fsspec:', fsspec.__version__)
try:
    import gcsfs
    print('gcsfs:', gcsfs.__version__)
except Exception as e:
    print('gcsfs optional nicht verfügbar:', e)
!apt-get clean
!rm -rf /var/lib/apt/lists/* /var/cache/apt/archives/*
!pip cache purge >/dev/null 2>&1 || true
total, used, free = shutil.disk_usage('/content')
print(f'Colab-Disk nach Setup: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')
print('Setup abgeschlossen')

In [ ]:
# Git-Befehle mit sauberem Python- und Shell-Arbeitsverzeichnis ausführen.
%cd /content
import os, pathlib, shutil, subprocess
BASE = '/content'
REPO = os.path.join(BASE, 'DanceMatch3')
BRANCH = 'feature/dance-style-training'
URL = 'https://github.com/zwirbel/DanceMatch3.git'
os.chdir(BASE)
os.environ['PWD'] = BASE

if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    print(f'{REPO} existiert, ist aber kein Git-Repository. Entferne es ...')
    shutil.rmtree(REPO)

if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], cwd=BASE, check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], cwd=BASE, check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], cwd=BASE, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, URL, REPO], cwd=BASE, check=True)

script = pathlib.Path(REPO, 'training', 'train_baseline.py')
if not script.is_file():
    raise FileNotFoundError(f'Trainingsskript nicht gefunden: {script}')
print('Code erfolgreich geladen:', REPO)
print('Branch:', subprocess.run(['git', '-C', REPO, 'branch', '--show-current'], cwd=BASE, check=True, capture_output=True, text=True).stdout.strip())


## Trainingsumfang auswählen

Der Datensatz wird **gestreamt**: Audiodaten werden einzeln abgerufen und verarbeitet, statt den gesamten Datensatz als lokale Dataset-Kopie zu speichern. Das Skript behält nur kompakte Merkmalsvektoren. Der projektspezifische Hugging-Face-Cache wird auch bei Fehlern bereinigt.

\`MAX_ITEMS = 0\` verarbeitet alle verfügbaren Beispiele (bis zu 698 Zeilen laut aktuellem Datensatzstand). Eine positive Zahl, zum Beispiel \`100\`, begrenzt den Lauf auf einen kurzen Funktionstest. Das Skript bricht vorsorglich ab, wenn weniger als 0.75 GiB freier Speicher vorhanden sind.

In [ ]:
MAX_ITEMS = 0  # 0 = vollständiger Streaming-Lauf; positive Zahl = kurzer Funktionstest
DATASET = 'Yi3852/MuCUE'
SPLIT = 'ballroom_genres'  # MuCUE hat die Dataset-Config 'default'; ballroom_genres ist ein Split
CACHE_DIR = '/content/dancematch_hf_cache'
OUT_DIR = f'{REPO}/training/artifacts'\nSUPPLEMENT_DIR = '/content/dancematch_supplement'
os.makedirs(OUT_DIR, exist_ok=True)
print('MAX_ITEMS =', MAX_ITEMS, '| DATASET =', DATASET, '| SPLIT =', SPLIT)
print('Isolierter, temporärer Cache:', CACHE_DIR)

In [ ]:
# Optionale Zusatzdaten: ZIP muss Klassenordner direkt im ZIP-Root enthalten.
# Beispiel: Bachata/clip01.mp3, Discofox/clip01.mp3, Salsa/clip01.mp3
# Verwende nur eigene, frei lizenzierte oder ausdrücklich zur Nutzung freigegebene Aufnahmen.
# Für einen reinen Baseline-Lauf ADD_SUPPLEMENTAL_AUDIO auf False setzen.
ADD_SUPPLEMENTAL_AUDIO = True
from pathlib import Path
import io, zipfile, shutil, os
SUPPLEMENT_DIR = Path('/content/dancematch_supplement')

if ADD_SUPPLEMENTAL_AUDIO:
    from google.colab import files
    if SUPPLEMENT_DIR.exists():
        shutil.rmtree(SUPPLEMENT_DIR)
    SUPPLEMENT_DIR.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('Bitte ein ZIP mit Klassenordnern hochladen, z. B. Bachata/, Discofox/ und Salsa/.')
    for name in zip_names:
        with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
            root = SUPPLEMENT_DIR.resolve()
            for entry in archive.infolist():
                target = (SUPPLEMENT_DIR / entry.filename).resolve()
                if target != root and root not in target.parents:
                    raise ValueError('Unsicherer Pfad im ZIP: ' + entry.filename)
                if entry.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(archive.read(entry))
    print('Zusatzdaten entpackt nach:', SUPPLEMENT_DIR)
    print('Gefundene Klassenordner:', [p.name for p in SUPPLEMENT_DIR.iterdir() if p.is_dir()])
else:
    print('Zusatzdaten-Upload übersprungen; fehlende Klassen bleiben im Training.')


In [ ]:
# Alte Modell- und Bewertungsdateien entfernen, damit ein fehlgeschlagener Lauf
# nicht versehentlich Ergebnisse eines früheren Laufs als aktuell ausgibt.
from pathlib import Path
ARTIFACTS = Path('/content/DanceMatch3/training/artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name in ('dance_baseline.joblib', 'evaluation.json', 'run_manifest.json'):
    old_file = ARTIFACTS / name
    if old_file.exists():
        old_file.unlink()
        print('Altes Ergebnis entfernt:', old_file)

%cd /content/DanceMatch3/training
!DANCEMATCH_CACHE_DIR="$CACHE_DIR" HF_HUB_DISABLE_TELEMETRY=1 python train_baseline.py --dataset "$DATASET" --split "$SPLIT" --max-items "$MAX_ITEMS" --out artifacts

In [ ]:
# Nur die Ergebnisse des soeben gestarteten Laufs anzeigen und als eindeutig benannte ZIP exportieren.
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import zipfile
from google.colab import files

artifacts = Path('/content/DanceMatch3/training/artifacts')
model_path = artifacts / 'dance_baseline.joblib'
evaluation_path = artifacts / 'evaluation.json'

# Nicht fortfahren, wenn das Training keine neuen Ergebnisdateien erzeugt hat.
missing = [str(p) for p in (model_path, evaluation_path) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        'Aktuelle Trainingsergebnisse fehlen: ' + ', '.join(missing) +
        '. Bitte zuerst die Trainingszelle erfolgreich ausführen.'
    )

report = json.loads(evaluation_path.read_text(encoding='utf-8'))
now = datetime.now(timezone.utc)
manifest = {
    'created_at_utc': now.isoformat(timespec='seconds'),
    'model_file': model_path.name,
    'model_size_bytes': model_path.stat().st_size,
    'model_modified_at_utc': datetime.fromtimestamp(
        model_path.stat().st_mtime, timezone.utc
    ).isoformat(timespec='seconds'),
    'evaluation_file': evaluation_path.name,
    'evaluation_size_bytes': evaluation_path.stat().st_size,
    'evaluation_modified_at_utc': datetime.fromtimestamp(
        evaluation_path.stat().st_mtime, timezone.utc
    ).isoformat(timespec='seconds'),
    'dataset': report.get('dataset'),
    'dataset_split': report.get('dataset_split'),
    'usable_tracks': report.get('usable_tracks'),
    'rows_seen': report.get('rows_seen'),
    'skipped_rows': report.get('skipped_rows'),
    'macro_f1': report.get('macro_f1'),
    'class_counts': report.get('class_counts'),\n    'missing_target_classes': report.get('missing_target_classes'),\n    'is_nine_class_model': report.get('is_nine_class_model'),\n    'supplemental_tracks': report.get('supplemental_tracks'),
}
manifest_path = artifacts / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('AKTUELLER TRAININGSLAUF')
print('Datensatz:', manifest['dataset'], '| Split:', manifest['dataset_split'])
print('Verwendete Musikstücke:', manifest['usable_tracks'])
print('Gesehene Zeilen:', manifest['rows_seen'], '| Übersprungen:', manifest['skipped_rows'])
print('Macro-F1:', manifest['macro_f1'])
print('Klassen:', json.dumps(manifest['class_counts'], ensure_ascii=False))\nprint('Fehlende Zielklassen:', manifest.get('missing_target_classes'))\nprint('Vollständiges 9-Klassen-Modell:', manifest.get('is_nine_class_model'))
for p in (model_path, evaluation_path, manifest_path):
    print(p, f'({p.stat().st_size:,} bytes)')

stamp = now.strftime('%Y%m%d-%H%M%S')
zip_path = f'/content/dancematch-training-results-{stamp}-UTC.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(model_path, model_path.name)
    z.write(evaluation_path, evaluation_path.name)
    z.write(manifest_path, manifest_path.name)

files.download(zip_path)
print('ZIP erstellt:', zip_path)

# Sicherheitsnetz: der Trainingsprozess bereinigt seinen Cache bereits im finally-Block.
cache_path = Path(CACHE_DIR)
if cache_path.exists():
    cache_bytes = sum(p.stat().st_size for p in cache_path.rglob('*') if p.is_file())
    shutil.rmtree(cache_path, ignore_errors=True)
    print(f'Übrig gebliebenen DanceMatch-Cache bereinigt: {cache_bytes / (1024**2):.1f} MiB.')
total, used, free = shutil.disk_usage('/content')
print(f'Colab-Disk nach Bereinigung: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')